# Task 3 — 80/10/10 single-fold model-selection check

Retrospective diagnostic on labeled Train: fit on 80%, select using the first 10%, inspect the second 10% once. All rows have appeared in previous research, so the final split is not independent confirmation. Test is not read.


In [1]:
from pathlib import Path
import sys,json,joblib
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'src'/'pipeline'/'save_model.py').exists())
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
from src.pipeline.random_seed import set_seed
from src.pipeline.save_model import save_model
from src.pipeline.evaluate_model import evaluate_model
SEED=set_seed()
import numpy as np,pandas as pd
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.metrics import fbeta_score,mean_absolute_error
from IPython.display import display
SRC=ROOT/'src'


In [2]:
TASK=SRC/'task3';SAVE=TASK/'save';DATA='eda_combined_v1'
card=json.loads((TASK/'data'/'eda'/f'{DATA}.json').read_text(encoding='utf-8'))
train=pd.read_parquet(SRC/card['splits']['train']['path']);y=train.spending_30d.astype(float)
bands=pd.cut(y,bins=[-1,0,100,1000,10000,50000,np.inf],labels=False)
fit_pos,temp_pos=train_test_split(np.arange(len(y)),test_size=.2,stratify=bands,random_state=SEED)
select_pos,check_pos=train_test_split(temp_pos,test_size=.5,stratify=bands.iloc[temp_pos],random_state=SEED)
assert (len(fit_pos),len(select_pos),len(check_pos))==(83200,10400,10400)
X=train.drop(columns=['player_id','spending_30d'])
old=joblib.load(SAVE/'t3_gate_blend_cv5_v2.joblib')
from src.task3.models.spending_models import PositiveGate,PrefitBlend,WeightedPair
from sklearn.compose import TransformedTargetRegressor
gate=clone(old.first_model.occurrence_model)
cat=clone(old.second_model.catboost_model)
lgb=clone(old.second_model.lightgbm_model)
gate.fit(X.iloc[fit_pos],(y.iloc[fit_pos]>0).astype(int))
cat.fit(X.iloc[fit_pos],y.iloc[fit_pos]);lgb.fit(X.iloc[fit_pos],y.iloc[fit_pos])
continuous=PrefitBlend(cat,lgb,.4)
positive_pos=fit_pos[y.iloc[fit_pos].to_numpy()>0]
results={}
def record(name,source,amount):
 paths=[SAVE/f'{name}.joblib',SAVE/f'{name}.json',SAVE/f'{name}-evaluation.json']
 if any(p.exists() for p in paths):
  assert all(p.exists() for p in paths),f'incomplete {name}'
  model=joblib.load(paths[0]);saved=json.loads(paths[1].read_text(encoding='utf-8'))
  assert saved['data_name']==DATA and saved['card']['source_model']==source
 else:
  model=WeightedPair(PositiveGate(gate,amount,.5),continuous,.8)
  save_model(task=3,name=name,model=model,data_name=DATA,
   card={'idea':'80/10/10 retrospective selection check','source_model':source,'seed':SEED,
         'fit_rows':len(fit_pos),'selection_rows':len(select_pos),'check_rows':len(check_pos)})
 select_pred=model.predict(X.iloc[select_pos]);check_pred=model.predict(X.iloc[check_pos])
 selection=mean_absolute_error(y.iloc[select_pos],select_pred)
 check=mean_absolute_error(y.iloc[check_pos],check_pred)
 if not paths[2].exists():
  evaluate_model(task=3,name=name,score=float(check),metric_name='MAE_thb_split801010_check',
   higher_is_better=False,card={'split':'retrospective single-fold 80/10/10 check',
   'selection_MAE_thb':float(selection),'seed':SEED})
 else:
  saved=json.loads(paths[2].read_text(encoding='utf-8'));assert abs(saved['score']-check)<1e-6
 results[name]={'selection':selection,'check':check,'pred':check_pred}
 print(name,'selection MAE',round(selection,2),'check MAE',round(check,2))


## Prior fixed gate/blend baseline
The occurrence gate and continuous models were fitted only on the 80% split.


In [3]:
name='t3_split801010_baseline_v1'
if not (SAVE/f'{name}.joblib').exists():
 amount_base=clone(old.first_model.positive_amount_model);amount_base.fit(X.iloc[positive_pos],y.iloc[positive_pos])
else: amount_base=None
record(name,'t3_gate_blend_cv5_v2',amount_base)


t3_split801010_baseline_v1 selection MAE 2834.96 check MAE 2855.23


## Log-MAE positive amount
The gate and continuous components remain fixed.


In [4]:
name='t3_split801010_log_mae_v1'
if not (SAVE/f'{name}.joblib').exists():
 amount_log=clone(joblib.load(SAVE/'t3_log_positive_amount_cv5_v5.joblib').first_model.positive_amount_model)
 amount_log.fit(X.iloc[positive_pos],y.iloc[positive_pos])
else: amount_log=None
record(name,'t3_log_positive_amount_cv5_v5',amount_log)


t3_split801010_log_mae_v1 selection MAE 2766.04 check MAE 2795.84


## Prespecified half blend
The second log-RMSE amount model is fitted on the same 80% rows.


In [5]:
name='t3_split801010_halfblend_v1'
if not (SAVE/f'{name}.joblib').exists():
 if amount_log is None: amount_log=joblib.load(SAVE/'t3_split801010_log_mae_v1.joblib').first_model.positive_amount_model
 amount_rmse=clone(joblib.load(SAVE/'t3_log_rmse_amount_cv5_v7.joblib').first_model.positive_amount_model)
 amount_rmse.fit(X.iloc[positive_pos],y.iloc[positive_pos])
 amount_half=PrefitBlend(amount_log,amount_rmse,.5)
else: amount_half=None
record(name,'t3_log_amount_halfblend_cv5_v7',amount_half)


t3_split801010_halfblend_v1 selection MAE 2761.93 check MAE 2792.59


## Selection versus check
MAE is in THB; it is not competition NMAE.


In [6]:
table=pd.DataFrame([{'name':name,'selection':v['selection'],'check':v['check']} for name,v in results.items()])
table['selection_rank']=table.selection.rank(ascending=True,method='min').astype(int)
table['check_rank']=table.check.rank(ascending=True,method='min').astype(int)
display(table.sort_values('selection_rank').round(2))
chosen=table.sort_values('selection_rank').iloc[0]['name'];best=table.sort_values('check_rank').iloc[0]['name']
print('Selected on 10%:',chosen,'; best on check 10%:',best)
for a,b in [('t3_split801010_log_mae_v1','t3_split801010_halfblend_v1'),('t3_split801010_baseline_v1','t3_split801010_log_mae_v1')]:
 pa,pb=results[a]['pred'],results[b]['pred']
 print(a,'vs',b,'mean absolute prediction gap',round(np.mean(np.abs(pa-pb)),2),
       'gap >=1000 THB',int((np.abs(pa-pb)>=1000).sum()),'of',len(pa))
for group,mask in [('zero',y.iloc[check_pos].to_numpy()==0),('positive',y.iloc[check_pos].to_numpy()>0),('tail_50k',y.iloc[check_pos].to_numpy()>=50000)]:
 print(group,'rows',int(mask.sum()),{n:round(mean_absolute_error(y.iloc[check_pos].to_numpy()[mask],v['pred'][mask]),2) for n,v in results.items()})


,name,selection,check,selection_rank,check_rank
2,t3_split801010_halfblend_v1,2761.93,2792.59,1,1
1,t3_split801010_log_mae_v1,2766.04,2795.84,2,2
0,t3_split801010_baseline_v1,2834.96,2855.23,3,3


Selected on 10%: t3_split801010_halfblend_v1 ; best on check 10%: t3_split801010_halfblend_v1
t3_split801010_log_mae_v1 vs t3_split801010_halfblend_v1 mean absolute prediction gap 183.36 gap >=1000 THB 648 of 10400
t3_split801010_baseline_v1 vs t3_split801010_log_mae_v1 mean absolute prediction gap 465.86 gap >=1000 THB 1128 of 10400
zero rows 5005 {'t3_split801010_baseline_v1': 1712.23, 't3_split801010_log_mae_v1': 1699.58, 't3_split801010_halfblend_v1': 1711.85}
positive rows 5395 {'t3_split801010_baseline_v1': 3915.61, 't3_split801010_log_mae_v1': 3812.86, 't3_split801010_halfblend_v1': 3795.2}
tail_50k rows 848 {'t3_split801010_baseline_v1': 19862.43, 't3_split801010_log_mae_v1': 19531.55, 't3_split801010_halfblend_v1': 19374.19}


## Assessment

The fixed half blend ranks first on both 10% splits, by 4.11 and 3.25 THB MAE over log-MAE. Its gain is concentrated in positive and high-spend rows, while zero-target MAE worsens by 12.27 THB on the check split. The persistent direction is useful but small; keep the scored full-data log-MAE submission as the practical reference until a controlled external result exists.
